# Manutenção preditiva de CO₂ - SpaceStay (Parte 6, ML light touch)

Ideia simples: em vez de só reagir quando o CO₂ estoura, **prever** qual módulo está
**tendendo a estourar** e sinalizar manutenção preventiva antes do alerta crítico.

- Modelo: **árvore de decisão** (scikit-learn), dentro do conteúdo do curso.
- Entrada: features de tendência (drift) do CO₂ por módulo.
- Saída: um risco por módulo (offline; a API não chama isso ao vivo).

Roda no Google Colab (pandas, numpy e scikit-learn já vêm instalados).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)

## 1) Dados

Cada amostra é uma janela de observação de um módulo. Usamos features de **tendência**,
que é o que indica um problema chegando, e não a leitura crua:

- `co2_atual`: última leitura de CO₂ (ppm)
- `co2_slope`: inclinação (quão rápido o CO₂ sobe, ppm por leitura)
- `min_acima_800`: há quantos minutos o CO₂ está acima de 800 ppm

Rótulo `vai_estourar`: 1 se o módulo deve passar de 2000 ppm em breve.

Aqui geramos dados sintéticos para o exemplo. Na prática, dá para exportar a tabela
`sensor_readings` do banco (Parte 1) em CSV e calcular essas mesmas features.

In [ ]:
n = 400
co2_atual     = np.random.uniform(400, 1900, n)
co2_slope     = np.random.uniform(-5, 60, n)
min_acima_800 = np.random.uniform(0, 120, n)

# Regra "oculta" usada só para rotular os dados de exemplo (o modelo aprende a partir dela).
risco_real = co2_atual + co2_slope * 8 + min_acima_800 * 2
vai_estourar = (risco_real > 2200).astype(int)

df = pd.DataFrame({
    "co2_atual": co2_atual,
    "co2_slope": co2_slope,
    "min_acima_800": min_acima_800,
    "vai_estourar": vai_estourar,
})
print("amostras:", len(df), "| positivos:", int(df.vai_estourar.sum()))
df.head()

## 2) Treino

In [ ]:
X = df[["co2_atual", "co2_slope", "min_acima_800"]]
y = df["vai_estourar"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

print("Acuracia no teste:", round(accuracy_score(y_test, modelo.predict(X_test)), 3))

## 3) Risco por módulo (agora)

Aplicamos o modelo ao estado atual de cada módulo. No exemplo, a Cupola está com o CO₂
subindo rápido e a Aurora está estável.

In [ ]:
agora = pd.DataFrame(
    {"co2_atual": [1850, 600], "co2_slope": [45, 2], "min_acima_800": [90, 0]},
    index=["Cupola Suite 3", "Aurora Suite 1"],
)

risco = modelo.predict_proba(agora)[:, 1]
for modulo, p in zip(agora.index, risco):
    estado = "manutencao preventiva" if p >= 0.5 else "ok"
    print(f"{modulo}: risco de estourar {p*100:.0f}% ({estado})")

## 4) Conclusão

Com três features simples de tendência, uma árvore de decisão já separa os módulos que
caminham para um problema dos que estão estáveis. É um "light touch": análise offline que
aponta o caminho para uma manutenção preditiva do habitat, sem a API precisar chamar o
modelo em tempo real.